In [25]:
pip install pyodbc


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import pyodbc

# Use the absolute path to the .dylib file found in step 1
driver_path = '/opt/homebrew/lib/libmsodbcsql.17.dylib' 

conn_str = (
    f'DRIVER={{{driver_path}}};' # Note the triple curly braces for f-string + SQL format
    'SERVER=10.96.189.36;'
    'DATABASE=SCS3g3;'
    'UID=scs3g3;PWD=P@ssw0rd!;'
)

try:
    # 2. Establish connection
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    # 3. Execute a query
    query = "SELECT * FROM Warehouses"
    cursor.execute(query)
    
    # 4. Fetch and print results
    for row in cursor:
        print(row)

except Exception as e:
    print(f"Error connecting to SQL Server: {e}")

Error connecting to SQL Server: <class 'pyodbc.Error'> returned a result with an exception set


In [22]:
## Clean Up
cursor.execute("""
DROP TABLE IF EXISTS Supplier_Shipment;
DROP TABLE IF EXISTS Shipment_PO;
DROP TABLE IF EXISTS Shipment_Warehouse;
DROP TABLE IF EXISTS Ship_Item;
DROP TABLE IF EXISTS Order_Item;
DROP TABLE IF EXISTS Shipments;
DROP TABLE IF EXISTS Purchase_Order_Client;
DROP TABLE IF EXISTS Purchase_Order;
DROP TABLE IF EXISTS Items;
DROP TABLE IF EXISTS Products;
DROP TABLE IF EXISTS Clients;
DROP TABLE IF EXISTS Suppliers;
DROP TABLE IF EXISTS Warehouses;
""")


OperationalError: ('08S01', '[08S01] [Microsoft][ODBC Driver 17 for SQL Server]Communication link failure (0) (SQLExecDirectW)')

In [15]:
## Create tables

cursor.execute("""
-- 1. Independent Tables
CREATE TABLE Products (
    Product_ID INT PRIMARY KEY,
    Name VARCHAR(100) NOT NULL,
    Brand VARCHAR(50),
    Cost DECIMAL(18, 2) NOT NULL,
    Category VARCHAR(50),
    Price DECIMAL(18, 2) NOT NULL,
    Length DECIMAL(10, 2),
    Width DECIMAL(10, 2),
    Height DECIMAL(10, 2),
    Handling_reuirements VARCHAR(255)
);

CREATE TABLE Warehouses (
    WarehouseID INT PRIMARY KEY,
    Name VARCHAR(100) NOT NULL,
    City VARCHAR(50),
    Country VARCHAR(50)
);

CREATE TABLE Clients (
    ClientID INT PRIMARY KEY,
    CompanyName VARCHAR(100) NOT NULL
);

CREATE TABLE Suppliers (
    SupplierID INT PRIMARY KEY,
    Name VARCHAR(100) NOT NULL,
    Country VARCHAR(50),
    PaymentTerms VARCHAR(255),
    LeadTime INT
);

CREATE TABLE Purchase_Order (
    OrderID INT PRIMARY KEY,
    Status VARCHAR(20) NOT NULL,
    OrderDate DATE NOT NULL,
    Value DECIMAL(18, 2) NOT NULL
);

-- 2. Level 1 Dependencies
CREATE TABLE Items (
    Serial_No INT PRIMARY KEY,
    Product_ID INT REFERENCES Products(Product_ID)
);

CREATE TABLE Purchase_Order_Client (
    OrderID INT REFERENCES Purchase_Order(OrderID),
    ClientID INT REFERENCES Clients(ClientID),
    PRIMARY KEY (OrderID, ClientID)
);

CREATE TABLE Shipments (
    ShipmentID INT PRIMARY KEY,
    PurchaseOrderID INT REFERENCES Purchase_Order(OrderID), -- Optional if using Shipment_PO
    Ex_ArrivalDate DATE NOT NULL,
    Actual_ArrivalDate DATE,
    Ex_ShippedLocation VARCHAR(100),
    TrackingNumber VARCHAR(50),
    ShippedDate DATE
);

-- 3. Level 2 Dependencies (Junction Tables)
CREATE TABLE Order_Item (
    Serial_No INT NOT NULL,
    OrderID INT REFERENCES Purchase_Order(OrderID),
    ExpectedDeliveryDate DATE NOT NULL,
    Ordered_Qty INT NOT NULL,
    UnitPrice DECIMAL(18, 2) NOT NULL,
    PRIMARY KEY (Serial_No, OrderID)
);

CREATE TABLE Ship_Item (
    Serial_No INT NOT NULL REFERENCES Items(Serial_No),
    OrderID INT REFERENCES Purchase_Order(OrderID),
    ActualDeliveryDate DATE NOT NULL,
    Delivered_Qty INT NOT NULL,
    PRIMARY KEY (Serial_No, OrderID)
);

CREATE TABLE Supplier_Shipment (
    ShipmentID INT REFERENCES Shipments(ShipmentID),
    SupplierID INT REFERENCES Suppliers(SupplierID),
    PRIMARY KEY (ShipmentID, SupplierID)
);

CREATE TABLE Shipment_Warehouse (
    ShipmentID INT REFERENCES Shipments(ShipmentID),
    WarehouseID INT REFERENCES Warehouses(WarehouseID),
    PRIMARY KEY (ShipmentID, WarehouseID)
);

CREATE TABLE Shipment_PO (
    ShipmentID INT REFERENCES Shipments(ShipmentID),
    OrderID INT REFERENCES Purchase_Order(OrderID),
    PRIMARY KEY (ShipmentID, OrderID)
);
""")

In [16]:
## insert data
cursor.execute("""
-- Columns: ID, Name, Brand, Cost, Category, Price, Length, Width, Height, Handling
INSERT INTO Products (
    Product_ID, Name, Brand, Cost, Category, 
    Price, Length, Width, Height, Handling_reuirements
) VALUES 
(101, 'Laptop Pro', 'Tech', 800.00, 'Computing', 1200.00, 35.0, 25.0, 2.0, 'Fragile'),
(102, 'Monitor 4K', 'View', 200.00, 'Display', 350.00, 60.0, 40.0, 5.0, 'Fragile'),
(103, 'Phone X', 'Mobile', 500.00, 'Mobile', 900.00, 15.0, 7.0, 0.8, 'Electronic'),
(104, 'Desk Chair', 'Ergo', 100.00, 'Furniture', 250.00, 60.0, 60.0, 120.0, 'None');

INSERT INTO Clients VALUES (1, 'Alpha Corp'), (2, 'Beta Ltd'), (3, 'Gamma Inc'), (4, 'Delta Co');

INSERT INTO Warehouses VALUES 
(501, 'SG-Main', 'Singapore', 'Singapore'),
(502, 'LA-West', 'Los Angeles', 'USA');

INSERT INTO Suppliers VALUES 
(901, 'SG-Direct', 'Singapore', 'Net 30', 5),
(902, 'Global-Log', 'USA', 'Net 60', 15);

-- Orders
INSERT INTO Purchase_Order VALUES 
(2007, 'Completed', '2024-05-10', 1500.00),
(2008, 'Completed', '2024-05-22', 2100.00),
(2009, 'Completed', '2024-11-05', 4000.00),
(2010, 'Completed', '2024-12-12', 5500.00),
(2011, 'Completed', '2025-05-15', 3200.00),
(2012, 'Completed', '2025-05-28', 1800.00),
(2015, 'Completed', '2025-08-15', 6000.00),
(2017, 'Completed', '2025-11-25', 3000.00);

-- Links
INSERT INTO Purchase_Order_Client VALUES (2007, 1), (2008, 2), (2009, 3), (2010, 4), (2011, 1), (2012, 2), (2015, 1), (2017, 3);

-- Shipments
INSERT INTO Shipments (ShipmentID, Ex_ArrivalDate, Actual_ArrivalDate, Ex_ShippedLocation) VALUES 
(3005, '2024-06-10', '2024-06-12', 'Port A'),
(3006, '2024-06-20', '2024-07-01', 'Port B'),
(3007, '2024-12-05', '2024-12-10', 'Port C'),
(3008, '2025-01-15', '2025-01-20', 'Port D'),
(3009, '2025-06-15', '2025-06-18', 'Port E'),
(3010, '2025-07-01', '2025-07-05', 'Port F'),
(3011, '2025-09-15', '2025-10-15', 'Port G'),
(3012, '2025-12-20', '2025-12-22', 'Port H');

-- Map Shipments to Warehouses & Orders
INSERT INTO Shipment_Warehouse VALUES (3005, 501), (3007, 501), (3009, 501), (3011, 501), (3006, 502), (3008, 502), (3010, 502), (3012, 502);
INSERT INTO Shipment_PO VALUES (3005, 2007), (3006, 2008), (3007, 2009), (3008, 2010), (3009, 2011), (3010, 2012), (3011, 2015), (3012, 2017);
""")

# Question 1

For each warehouse, find its' top three clients (those who brought in the most amount of business in
dollar terms to the warehouse).

In [ ]:
query = """WITH ClientTotals AS (
    SELECT
        W.Name AS Warehouse,
        C.CompanyName AS Client,
        SUM(PO.Value) AS TotalBusinessValue,
        ROW_NUMBER() OVER(PARTITION BY W.WarehouseID ORDER BY SUM(PO.Value) DESC) AS Rank
    FROM Purchase_Order PO
    JOIN Purchase_Order_Client POC ON PO.OrderID = POC.OrderID
    JOIN Clients C ON POC.ClientID = C.ClientID
    -- THE BRIDGE: Link PO to Shipment via the mapping table
    JOIN Shipment_PO SPO ON PO.OrderID = SPO.OrderID
    JOIN Shipments S ON SPO.ShipmentID = S.ShipmentID
    -- THE BRIDGE: Link Shipment to Warehouse
    JOIN Shipment_Warehouse SW ON S.ShipmentID = SW.ShipmentID
    JOIN Warehouses W ON SW.WarehouseID = W.WarehouseID
    GROUP BY W.WarehouseID, W.Name, C.ClientID, C.CompanyName
)
SELECT Warehouse, Client, TotalBusinessValue
FROM ClientTotals
WHERE Rank <= 3
ORDER BY Warehouse, Rank;
"""

cursor.execute(query)

for row in cursor:
    print(row)

In [23]:
conn.commit()
conn.close()

OperationalError: ('08S01', '[08S01] [Microsoft][ODBC Driver 17 for SQL Server]Communication link failure (-2147467259) (SQLEndTran)')